In [61]:
import pandas as pd
df=pd.read_parquet("/Users/saratramontana/Documents/cartella tirocinio/test_segmentation_model/fake_train_fold_0_seed_0.parquet")
df.head()

,clinical_case,item,frame,image,gt_mask,risk_class
0,-HTqLW7UkQH0F8KfsPItp,1W6gtZAwVEJIUBOGQGEpb,00000,"b'\xd7BK\xecHh""m\xbd\xdf\x8bx\xc52z:\x84r\x16\...",b'(\xec\x1f3\x11\xc3\x8eh=n\xc3\xc2M?~6\x18\xd...,malignant
319,-HTqLW7UkQH0F8KfsPItp,p-1DhH_ycRLX4kMdPC1B9,00000,b'#\xb6\x9e\xb7\x16\xe6\xb1\x83\xdc\xedz\xa6\x...,b'\xe7r\xb5wB0~\x99\x08B\xb7\xd9A\x15Ie\x03;\x...,malignant
0,-Nd6pZbGA0TACh9Kv1Lqk,ylaa_dN9PExKFTgOtaL0D,00000,b'\xe0_\x86V{\xb3\xab\xc9\xb4\xfa\xa0Lv\xac^\x...,b'0\xd5\x83l\xda\xb4\x97vdJ\xd5\xe9\xd8\x15\xe...,malignant
5262617663,-Nd6pZbGA0TACh9Kv1Lqk,ylaa_dN9PExKFTgOtaL0D,00002,b'\xf4\xc4\tp\x11\xea~\x0c\xbd\xe3\xd8\xd7\xcf...,b'9\xef{\xbb\xe7V\xf2{\xb7\xaa|?\x00L\xd1]Tja:...,malignant
1,-Nd6pZbGA0TACh9Kv1Lqk,ylaa_dN9PExKFTgOtaL0D,00005,b'F\x07\xb2d.\xe0\xd7\xdeWu\xf6\x04\xf4\xd6J\x...,b'\xac\x9e\x07G\xb1H\xd0y\x81!\xeed\xf0\x98z\x...,malignant


In [62]:
#%pip install wandb

In [63]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import lightning as L
import segmentation_models_pytorch as smp
from lightning.pytorch.loggers import WandbLogger
import wandb


In [64]:
class MyDataset(Dataset):
    def __init__(self,df,num_samples=100, image_size=256):
        self.df=df.reset_index(drop=True) #to have a sortered index: (0,1,2 ecc..)
        self.num_samples=num_samples
        self.image_size=image_size

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]

        image = torch.tensor(np.frombuffer(row["image"], dtype=np.uint8).reshape(224,224)).float()/255 #frombuffer to convert bytes into integers from 0 to 255, then float because pytorch works with float
        image = image.unsqueeze(0)

        seg_mask = torch.tensor(np.frombuffer(row["gt_mask"], dtype=np.uint8).reshape(224,224)).float()/255
        seg_mask = (seg_mask > 0.5).float().unsqueeze(0) #First a boolean mask, then with float it becomes a binary mask and with unsqueeze I add a dimension

        cls_label = torch.tensor(1 if row["risk_class"]=="malignant" else 0)

        return image, seg_mask, cls_label
    
    



In [65]:
# train_dataset = MyDataset(df,num_samples=100, image_size=256)
# val_dataset = MyDataset(df, num_samples=20, image_size=256)

# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True) #batch size of 8 means that the model will process 8 samples at a time.
# val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
#!pip install scikit-learn
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
train_dataset = MyDataset(train_df)
val_dataset = MyDataset(val_df)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [66]:
batch = next(iter(train_loader))
images, seg_masks, cls_labels = batch

print(images.shape)
print(seg_masks.shape)
print(cls_labels.shape)

torch.Size([8, 1, 224, 224])
torch.Size([8, 1, 224, 224])
torch.Size([8])


In [67]:
class LitSegClsModel(L.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters() #saves the learning rate as a hyperparameter, which can be accessed later via self.hparams.lr

        aux_params = {
            "pooling": "avg",
            "dropout": 0.5,
            "activation": None,
            "classes": 2,
        }

        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=1,
            classes=1,
            aux_params=aux_params
        )

        self.seg_criterion = nn.BCEWithLogitsLoss()
        self.cls_criterion = nn.CrossEntropyLoss()

    def forward(self, x): #the forward method defines how the model processes input data and produces output
        seg_logits, cls_logits = self.model(x)
        return seg_logits, cls_logits

    def training_step(self, batch, batch_idx):
        images, seg_masks, cls_labels = batch #batch is a tuple containing the input images, the corresponding segmentation masks, and the classification labels. 

        seg_logits, cls_logits = self(images) #self(images) is equivalent to self.forward(images), it calls the forward method of the model to get the segmentation and classification logits.

        seg_loss = self.seg_criterion(seg_logits, seg_masks)
        cls_loss = self.cls_criterion(cls_logits, cls_labels)

        cls_preds=torch.argmax(cls_logits,dim=1)
        cls_acc=(cls_preds==cls_labels).float().mean()

        loss = seg_loss + cls_loss

        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True) #on_step=True → registra ogni batch, on_epoch=True → registra anche la media sull’epoca
        self.log("train_seg_loss", seg_loss,  on_step=True, on_epoch=True)
        self.log("train_cls_loss", cls_loss,  on_step=True, on_epoch=True)
        self.log("train_cls_acc", cls_acc, prog_bar=True, on_step=True, on_epoch=True )

        return loss

    def validation_step(self, batch, batch_idx):
        images, seg_masks, cls_labels = batch

        seg_logits, cls_logits = self(images) #seg_logits and cls_logits are the raw output of the model, they represent the unnormalized scores for each class in the segmentation and classification tasks, respectively.

        seg_loss = self.seg_criterion(seg_logits, seg_masks)
        cls_loss = self.cls_criterion(cls_logits, cls_labels)

        loss = seg_loss + cls_loss

        cls_preds = torch.argmax(cls_logits, dim=1)
        cls_acc = (cls_preds == cls_labels).float().mean()

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_seg_loss", seg_loss, on_step=False, on_epoch=True)
        self.log("val_cls_loss", cls_loss, on_step=False, on_epoch=True)
        self.log("val_cls_acc", cls_acc, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr) #self.parameters() returns an iterator over the model's parameters, which are the weights and biases that the optimizer will update during training. lr=self.hparams.lr sets the learning rate for the optimizer, which controls how much the model's parameters are updated in response to the computed gradients.
        return optimizer
    
    

In [68]:
# model = LitSegClsModel(lr=1e-3)
# wandb_logger = WandbLogger(
#     project="test-segmentation-model",   # scegli il nome del progetto
#     name="run_prova_1"                   # nome del singolo esperimento
# )
# trainer = L.Trainer(
#     max_epochs=2,
#     limit_train_batches=5, #only 5 batches for each epoch
#     limit_val_batches=2,
#     log_every_n_steps=1,
#     logger=wandb_logger) #it means that the trainer will log the training metrics after every batch, which is useful for monitoring the training process closely
# #Now the training starts
# trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Version with for iteration

In [69]:


# learning_rates = [1e-2, 1e-3, 1e-4]

# for lr in learning_rates:
#     wandb_logger = WandbLogger(
#         project="test-segmentation-model",
#         name=f"lr_{lr}"
#     )

#     wandb_logger.experiment.config["learning_rate"] = lr

#     model = LitSegClsModel(lr=lr)

#     trainer = L.Trainer(
#         max_epochs=2,
#         limit_train_batches=5,
#         limit_val_batches=2,
#         log_every_n_steps=1,
#         logger=wandb_logger
#     )

#     trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

#     wandb.finish()

Version with sweep 

In [70]:
def train():
    wandb.init()

    lr = wandb.config.lr

    wandb_logger = WandbLogger(
        project="test-segmentation-model"
    )

    model = LitSegClsModel(lr=lr)

    trainer = L.Trainer(
        max_epochs=2,
        limit_train_batches=5,
        limit_val_batches=2,
        log_every_n_steps=1,
        logger=wandb_logger
    )

    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

    wandb.finish()

In [71]:
sweep_config = {
    "method": "grid", #here you are only saying to experiment all the 3 lr
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "lr": {"values": [1e-2, 1e-3, 1e-4]}
    }
}

In [72]:
sweep_id = wandb.sweep(sweep_config, project="test-segmentation-model") #It creates a true sweep in the W&B account
print(sweep_id)

Create sweep with ID: 7txd4fia
Sweep URL: https://wandb.ai/sara-tramontana02-/test-segmentation-model/sweeps/7txd4fia
7txd4fia


In [73]:
wandb.agent(sweep_id, function=train, count=3) #with sweep_id you are telling which sweep to use
#in this case 3 runs since I have 3 lr

wandb: Agent Starting Run: bi3y6fdv with config:
wandb: 	lr: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/saratramontana/.netrc.


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ Unet              │ 24.4 M │ train │     0 │
│ 1 │ seg_criterion │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ cls_criterion │ CrossEntropyLoss  │      0 │ train │     0 │
└───┴───────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 24.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.4 M                                                                                               
Total estimated model params size (MB): 97                                                                         
Modules in train mode: 197                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

wandb: WARNING Config item 'lr' was locked by 'sweep' (ignored update).


...

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:2
1: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing 
the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=2` reached.


epoch,▁▁▁▁▁▁▁███████
train_cls_acc_epoch,█▁
train_cls_acc_step,▅▇▇▇█▄▁▅▇█
train_cls_loss_epoch,▁█
train_cls_loss_step,▃▄▂▁▂▃█▃▂▂
train_loss_epoch,▁█
train_loss_step,▃▄▂▁▂▃█▃▂▂
train_seg_loss_epoch,█▁
train_seg_loss_step,█▇▆▆▅▄▃▃▂▁
trainer/global_step,▁▂▃▃▄▄▄▅▆▆▇███
+4,...
